# step00c - QA/QC: Align CTD (Leah's slide graphs reproduced)

Reproduces the Align CTD assessment figures from the training slides, for our instrument
(SBE 19plus V2), so they can be shown and discussed.

Align CTD shifts a channel in time relative to pressure so temperature, conductivity and oxygen
describe the same parcel of water. Getting it right minimises salinity spiking and makes the
down and up profiles agree. We assess two channels the way the slides do:

| Figure | Channel | How it is judged |
|--------|---------|------------------|
| salinity | conductivity | 24 Hz dS/dt spiking and the 2 db down/up salinity difference at the sharp interface |
| oxygen | SBE 43 voltage | down/up agreement vs pressure and vs potential temperature |

**Instrument advances (19/19plus/19plus V2 row, not the 9plus row):** temperature **+0.5 s**
(fixed), oxygen **+3 to +7 s** (swept), conductivity **0 s** on the 19plus (the temperature advance
handles the C-T alignment; a small conductivity advance is swept here to confirm/tune it).

Metrics are measured in a window around the sharpest gradient (shaded on the plots), because
alignment spiking only shows at sharp interfaces. The advance is emulated as a time-shift and
salinity / potential temperature are computed here (PSS-78, UNESCO), so this needs no SBE binary and
scans a whole folder. It does not change any product: it tells you what to set in `02_alignctd.psa`.

> **Assumptions to discuss with Leah** are collected in the last section.


## 1. Imports, style, settings

In [ ]:
from __future__ import annotations
import re
from pathlib import Path
import numpy as np, pandas as pd

INTERACTIVE_ZOOM = False   # False (recommended): reliable static inline. True: ipympl zoom (needs ipympl)
import matplotlib
_backend = "widget" if INTERACTIVE_ZOOM else "inline"
try:
    get_ipython().run_line_magic("matplotlib", _backend)
except Exception:
    INTERACTIVE_ZOOM = False
    try: matplotlib.use("Agg")
    except Exception: pass
import matplotlib.pyplot as plt
from matplotlib import rcParams
try:
    from IPython.display import display
except Exception:
    def display(o): print(o)

rcParams.update({"figure.dpi":110,"savefig.dpi":300,"font.size":9,"axes.titlesize":10,
                 "axes.labelsize":9,"axes.linewidth":0.8,"xtick.direction":"in","ytick.direction":"in",
                 "xtick.labelsize":8,"ytick.labelsize":8,"legend.fontsize":7.5,"axes.grid":False,
                 "font.family":"DejaVu Sans"})

def _despine(ax):
    for s in ("top","right"): ax.spines[s].set_visible(False)

def finish(fig,out):
    """Save the figure and render it inline (reliably, as a static PNG unless zoom is on)."""
    saved=False
    for kw in ({"bbox_inches":"tight"},{}):
        try: fig.savefig(out,**kw); saved=True; break
        except Exception as e: print(f"    savefig {'tight' if kw else 'plain'} failed for {out.name}: {e}")
    if INTERACTIVE_ZOOM:
        try: fig.canvas.draw()
        except Exception: pass
        display(fig)
    else:
        display(fig); plt.close(fig)
    return out if saved else None


In [ ]:
# ---- paths / settings ----
CTD_ROOT  = Path(r"C:\Projects\ctd_pipeline")
CRUISE    = "P45_06"
# 24 Hz cnvs with conductivity, temperature, pressure (and SBE43 voltage). DatCnv or Filter output.
INPUT_DIR  = CTD_ROOT / "cruises" / CRUISE / "L1" / "CTD"
INPUT_GLOB = "*.cnv"
QC_PLOT_DIR = CTD_ROOT / "cruises" / CRUISE / "L2" / "CTD" / "qc_plots"; QC_PLOT_DIR.mkdir(parents=True, exist_ok=True)

FS = 24.0
T_ADVANCE = 0.5                                              # fixed (19plus V2)
COND_ADVANCES = [-0.25,-0.15,-0.10,-0.05,0.0,0.05,0.10,0.15,0.25]   # swept for the metric
O2_ADVANCES   = [3.0,3.5,4.0,4.5,5.0,5.5,6.0,6.5,7.0]               # swept for the metric
PLOT_COND_ADVANCES = [-0.15, 0.0, 0.15]                     # curves drawn on the salinity figure
PLOT_O2_ADVANCES   = [3.0, 5.0, 7.0]                        # curves drawn on the oxygen figure
PLOT_TOP_N = 3                                              # plot the N strongest-gradient casts
EXTRA_CASTS = ["P45_06_CTD_01", "P45_06_CTD_03"]           # always plot these (the interior -0.15 casts)

BIN_DB = 2.0
SURF_SKIP_DB = 8.0
GRAD_HALF_DB = 12.0
MIN_GRAD_DSDP = 0.005
C_STD = 4.2914                                              # C(35,15,0) in S/m
print("Align QC | input:", INPUT_DIR, "| T advance +", T_ADVANCE, "s")


## 2. Seawater (PSS-78 salinity, UNESCO potential temperature)

In [ ]:
def sal78(R, T68, P):
    a=[0.0080,-0.1692,25.3851,14.0941,-7.0261,2.7081]; b=[0.0005,-0.0056,-0.0066,-0.0375,0.0636,-0.0144]
    c=[0.6766097,2.00564e-2,1.104259e-4,-6.9698e-7,1.0031e-9]; d=[3.426e-2,4.464e-4,4.215e-1,-3.107e-3]
    e=[2.070e-5,-6.370e-10,3.989e-15]; R=np.asarray(R,float)
    Rp=1+(P*(e[0]+e[1]*P+e[2]*P*P))/(1+d[0]*T68+d[1]*T68*T68+(d[2]+d[3]*T68)*R)
    rt=c[0]+c[1]*T68+c[2]*T68**2+c[3]*T68**3+c[4]*T68**4
    Rt=R/(Rp*rt); Rt=np.maximum(Rt,1e-12); s=np.sqrt(Rt)
    dS=(T68-15)/(1+0.0162*(T68-15))*(b[0]+b[1]*s+b[2]*Rt+b[3]*Rt*s+b[4]*Rt**2+b[5]*Rt**2*s)
    return a[0]+a[1]*s+a[2]*Rt+a[3]*Rt*s+a[4]*Rt**2+a[5]*Rt**2*s+dS

def practical_salinity(C_Sm, T90, P):
    return sal78(np.asarray(C_Sm,float)/C_STD, np.asarray(T90,float)*1.00024, P)

def _adtg(S,T,P):
    a0,a1,a2,a3=3.5803e-5,8.5258e-6,-6.836e-8,6.6228e-10; b0,b1=1.8932e-6,-4.2393e-8
    c0,c1,c2,c3=1.8741e-8,-6.7795e-10,8.733e-12,-5.4481e-14; d0,d1=-1.1351e-10,2.7759e-12
    e0,e1,e2=-4.6206e-13,1.8676e-14,-2.1687e-16
    return (a0+(a1+(a2+a3*T)*T)*T+(b0+b1*T)*(S-35)
            +((c0+(c1+(c2+c3*T)*T)*T)+(d0+d1*T)*(S-35))*P+(e0+(e1+e2*T)*T)*P*P)

def potential_temperature(S,T90,P,PR=0.0):
    S=np.asarray(S,float); T=np.asarray(T90,float)*1.00024; P=np.asarray(P,float)
    H=PR-P; XK=H*_adtg(S,T,P); T=T+0.5*XK; Q=XK; P2=P+0.5*H
    XK=H*_adtg(S,T,P2); T=T+0.29289322*(XK-Q); Q=0.58578644*XK+0.121320344*Q
    XK=H*_adtg(S,T,P2); T=T+1.707106781*(XK-Q); Q=3.414213562*XK-4.121320344*Q
    P3=P2+0.5*H; XK=H*_adtg(S,T,P3); T=T+(XK-2.0*Q)/6.0
    return T/1.00024


## 3. Readers and helpers

In [ ]:
def read_cnv(path):
    txt=Path(path).read_text(encoding="latin-1",errors="replace").splitlines(); names,ds={},None
    nr=re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]",re.I)
    for i,l in enumerate(txt):
        s=l.strip()
        if s.startswith("#") or s.startswith("*"):
            m=nr.search(l)
            if m: names[int(m.group(1))]=m.group(2).strip()
            if s.upper().startswith("*END*"): ds=i+1; break
    if ds is None: return None
    cols=[names[k] for k in sorted(names)]
    rows=[l.split()[:len(cols)] for l in txt[ds:] if l.strip() and len(l.split())>=len(cols)]
    return pd.DataFrame(rows,columns=cols).apply(pd.to_numeric,errors="coerce").reset_index(drop=True)

def pick(df,*cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(x.lower() in c.lower() for x in cands): return c
    return None

def shift_advance(x,adv_s,fs=FS):
    x=np.asarray(x,float); n=len(x); idx=np.arange(n)
    return np.interp(idx+adv_s*fs, idx, x, left=x[0], right=x[-1])

def descent_start(pv,win=int(FS),min_drop=0.10):
    pv=np.asarray(pv,float)
    if pv.size==0 or np.isnan(pv).all(): return 0,0
    imax=int(np.nanargmax(pv))
    if imax<win+1: return 0,imax
    seg=pv[:imax+1]; s=pd.Series(seg).rolling(win,center=True,min_periods=1).median().to_numpy()
    fwd=np.full(len(s),np.nan); fwd[:len(s)-win]=s[win:]-s[:len(s)-win]
    desc=np.where(np.isnan(fwd),True,fwd>min_drop); i=imax
    while i>0 and desc[i-1]: i-=1
    return i,imax

def bin_profile(p,v,step=BIN_DB):
    d=pd.DataFrame({"p":p,"v":v}).dropna()
    if d.empty: return np.array([]),np.array([])
    b=(d["p"]/step).round().astype(int)*step; g=d.groupby(b)["v"].mean()
    return g.index.to_numpy(float), g.to_numpy(float)

def gradient_window(P,S,start,imax):
    pb,sb=bin_profile(P[start:imax+1],S[start:imax+1])
    if len(pb)<5: return SURF_SKIP_DB,float(np.nanmax(P)),None,0.0
    order=np.argsort(pb); pb,sb=pb[order],sb[order]; dsdp=np.abs(np.gradient(sb,pb))
    strength=float(np.nanpercentile(dsdp,95)); deep=pb>SURF_SKIP_DB
    if deep.sum()>=3 and strength>=MIN_GRAD_DSDP:
        centre=float(pb[deep][int(np.nanargmax(dsdp[deep]))])
        return centre-GRAD_HALF_DB,centre+GRAD_HALF_DB,centre,strength
    return SURF_SKIP_DB,float(np.nanmax(P)),None,strength

def _in(P,lo,hi): return (P>=lo)&(P<=hi)


## 4. Assessments (best advance per cast)

In [ ]:
def salinity_assessment(C,T,P,start,imax,lo,hi):
    Ta=shift_advance(T,T_ADVANCE); dn=np.zeros(len(P),bool); dn[start:imax+1]=True
    up=np.zeros(len(P),bool); up[imax:]=True; out={}
    for adv in COND_ADVANCES:
        S=practical_salinity(shift_advance(C,adv),Ta,P); m=dn&_in(P,lo,hi)&np.isfinite(S)
        std=float(np.nanstd(np.gradient(S[m]))) if m.sum()>3 else np.nan
        pbd,sbd=bin_profile(P[dn],S[dn]); pbu,sbu=bin_profile(P[up],S[up])
        common=np.intersect1d(pbd,pbu); common=common[(common>=lo)&(common<=hi)]
        if len(common):
            md_={p:s for p,s in zip(pbd,sbd)}; mu={p:s for p,s in zip(pbu,sbu)}
            updown=float(np.sqrt(np.nanmean([(md_[p]-mu[p])**2 for p in common])))
        else: updown=np.nan
        out[adv]={"S":S,"dSdt_std":std,"updown_rms":updown}
    best=min(out,key=lambda a: out[a]["dSdt_std"])
    return out,best,(best==min(COND_ADVANCES) or best==max(COND_ADVANCES))

def oxygen_assessment(O2v,S,T,P,start,imax,lo,hi):
    theta=potential_temperature(S,T,P); dn=np.zeros(len(P),bool); dn[start:imax+1]=True
    up=np.zeros(len(P),bool); up[imax:]=True; win=_in(P,lo,hi); out={}
    for adv in O2_ADVANCES:
        Oa=shift_advance(O2v,adv)
        thd,od=bin_profile(theta[dn&win],Oa[dn&win]); thu,ou=bin_profile(theta[up&win],Oa[up&win])
        common=np.intersect1d(np.round(thd,1),np.round(thu,1))
        if len(common):
            md_={round(t,1):o for t,o in zip(thd,od)}; mu={round(t,1):o for t,o in zip(thu,ou)}
            rms=float(np.sqrt(np.nanmean([(md_[t]-mu[t])**2 for t in common])))
        else: rms=np.nan
        out[adv]={"O2":Oa,"updown_rms":rms}
    valid={a:o for a,o in out.items() if np.isfinite(o["updown_rms"])}
    best=min(valid,key=lambda a: valid[a]["updown_rms"]) if valid else None
    return out,theta,best,(best is not None and (best==min(O2_ADVANCES) or best==max(O2_ADVANCES)))


## 5. Slide-style figures

Salinity: the four-panel from the conductivity slide (24 Hz salinity, 24 Hz dS/dt, 2 db salinity, 2 db down/up difference). Oxygen: O2 vs pressure and vs potential temperature, down (solid) and up (dashed). The gradient window is shaded.

In [ ]:
C_BEST="#D62728"

def plot_salinity(cast,C,T,P,start,imax,sal_out,best,lo,hi,centre):
    Ta=shift_advance(T,T_ADVANCE); dn=slice(start,imax+1)
    fig,ax=plt.subplots(1,4,figsize=(12.5,6.0),sharey=True)
    cmap=plt.cm.viridis(np.linspace(0,1,len(PLOT_COND_ADVANCES)))
    for c,adv in zip(cmap,PLOT_COND_ADVANCES):
        S=sal_out[adv]["S"]; Sd=S[dn]; Pd=P[dn]
        ax[0].plot(Sd,Pd,lw=0.5,color=c,alpha=0.9,label=f"{adv:+.2f}s")
        ax[1].plot(np.gradient(Sd),Pd,lw=0.5,color=c,alpha=0.9)
        pb,sb=bin_profile(Pd,Sd); ax[2].plot(sb,pb,lw=1.1,color=c)
    Sb=sal_out[best]["S"]; pbd,sbd=bin_profile(P[dn],Sb[dn]); pbu,sbu=bin_profile(P[imax:],Sb[imax:])
    common=np.intersect1d(pbd,pbu)
    if len(common):
        md_={p:s for p,s in zip(pbd,sbd)}; mu={p:s for p,s in zip(pbu,sbu)}
        ax[3].plot([md_[p]-mu[p] for p in common],common,lw=0.9,color="#333333"); ax[3].axvline(0,color="k",lw=0.5,alpha=0.4)
    for a in ax:
        if centre is not None: a.axhspan(lo,hi,color="#F4A24A",alpha=0.12)
        a.invert_yaxis(); _despine(a)
    ax[0].set_ylabel("pressure (db)")
    ax[0].set_title("24 Hz salinity"); ax[0].set_xlabel("salinity (PSU)")
    ax[1].set_title("24 Hz dS/dt"); ax[1].set_xlabel("dS/dt (PSU/scan)")
    ax[2].set_title(f"{BIN_DB:g} db salinity"); ax[2].set_xlabel("salinity (PSU)")
    ax[3].set_title(f"{BIN_DB:g} db down-up (best {best:+.2f}s)"); ax[3].set_xlabel("\u0394 salinity (PSU)")
    ax[0].legend(title="cond advance",loc="lower left",fontsize=7,title_fontsize=7)
    fig.suptitle(f"{cast}: Align CTD conductivity / salinity (T advance +{T_ADVANCE:g} s)",fontsize=11)
    fig.tight_layout(rect=[0,0,1,0.98])
    return finish(fig, QC_PLOT_DIR/f"{cast}__qc_align_salinity.png")

def plot_oxygen(cast,theta,P,start,imax,o2_out,best,lo,hi,centre):
    dn=slice(start,imax+1); up=slice(imax,len(P))
    fig,ax=plt.subplots(1,2,figsize=(9.5,6.0))
    cmap=plt.cm.plasma(np.linspace(0,0.85,len(PLOT_O2_ADVANCES)))
    for c,adv in zip(cmap,PLOT_O2_ADVANCES):
        O=o2_out[adv]["O2"]; lbl=f"{adv:g}s"+(" *best" if adv==best else "")
        ax[0].plot(O[dn],P[dn],lw=0.6,color=c,label=lbl); ax[0].plot(O[up],P[up],lw=0.6,color=c,ls="--",alpha=0.7)
        ax[1].plot(O[dn],theta[dn],lw=0.6,color=c); ax[1].plot(O[up],theta[up],lw=0.6,color=c,ls="--",alpha=0.7)
    if centre is not None: ax[0].axhspan(lo,hi,color="#F4A24A",alpha=0.12)
    ax[0].invert_yaxis(); _despine(ax[0]); _despine(ax[1])
    ax[0].set_ylabel("pressure (db)"); ax[0].set_xlabel("SBE 43 voltage"); ax[0].set_title("O2 vs pressure (down solid, up dashed)")
    ax[1].set_ylabel("potential temperature (\u00b0C)"); ax[1].set_xlabel("SBE 43 voltage"); ax[1].set_title("O2 vs potential temperature")
    ax[0].legend(loc="best",fontsize=7)
    fig.suptitle(f"{cast}: Align CTD oxygen (down/up should agree near sharp changes)",fontsize=11)
    fig.tight_layout(rect=[0,0,1,0.98])
    return finish(fig, QC_PLOT_DIR/f"{cast}__qc_align_oxygen.png")


## 6. Run all casts

In [ ]:
rows=[]; ALL={}
for f in sorted(Path(INPUT_DIR).glob(INPUT_GLOB)):
    df=read_cnv(f)
    if df is None: continue
    pcol=pick(df,"prdM","prDM","prM"); tcol=pick(df,"tv290C","t090C","tv290")
    ccol=pick(df,"c0S/m","cond0S/m","c0mS/cm","c_S/m","cond")
    if pcol is None or tcol is None or ccol is None:
        print(f"{f.name}: missing P/T/C, skipping"); continue
    ocol=pick(df,"sbeox0V","sbox0","oxygen raw","sbeoxV","sbeox0v")
    P=pd.to_numeric(df[pcol],errors="coerce").to_numpy(float)
    T=pd.to_numeric(df[tcol],errors="coerce").to_numpy(float)
    C=pd.to_numeric(df[ccol],errors="coerce").to_numpy(float)
    start,imax=descent_start(P)
    S0=practical_salinity(shift_advance(C,0.0),shift_advance(T,T_ADVANCE),P)
    lo,hi,centre,strength=gradient_window(P,S0,start,imax)
    sal_out,best_c,cb=salinity_assessment(C,T,P,start,imax,lo,hi)
    row={"cast":f.stem,"grad_strength":round(strength,4),"grad_centre_db":round(centre,1) if centre is not None else "",
         "gradient":"weak" if centre is None else "ok","best_cond_advance_s":best_c,
         "cond_flag":"BOUNDARY" if cb else "interior","dSdt_std_best":round(sal_out[best_c]["dSdt_std"],5)}
    o2_out=theta=best_o2=None; ob=False
    if ocol is not None:
        O2v=pd.to_numeric(df[ocol],errors="coerce").to_numpy(float)
        o2_out,theta,best_o2,ob=oxygen_assessment(O2v,S0,T,P,start,imax,lo,hi)
        row["best_o2_advance_s"]=best_o2 if best_o2 is not None else ""; row["o2_flag"]="BOUNDARY" if ob else "interior"
    rows.append(row)
    ALL[f.stem]=dict(C=C,T=T,P=P,start=start,imax=imax,sal_out=sal_out,best_c=best_c,
                     o2_out=o2_out,theta=theta,best_o2=best_o2,has_o2=ocol is not None,lo=lo,hi=hi,centre=centre)

if not rows:
    print("no usable casts found - check INPUT_DIR and column names")
else:
    summ=pd.DataFrame(rows).sort_values("grad_strength",ascending=False).reset_index(drop=True)
    summ.to_csv(QC_PLOT_DIR/"qc_align_summary.csv",index=False,encoding="utf-8-sig")
    display(summ)
    good=summ[(summ["gradient"]=="ok")&(summ["cond_flag"]=="interior")]
    if len(good):
        print("Conductivity interior (trustworthy) bests:",sorted(good["best_cond_advance_s"]),
              "| median",f"{np.median(good['best_cond_advance_s']):+.2f} s")
    else:
        print("Conductivity: no interior optimum -> keep 0 s.")
    # figures for the top-gradient casts plus the explicitly requested ones
    to_plot=list(dict.fromkeys(list(summ["cast"].head(PLOT_TOP_N))+[c for c in EXTRA_CASTS if c in ALL]))
    for cast in to_plot:
        c=ALL[cast]
        plot_salinity(cast,c["C"],c["T"],c["P"],c["start"],c["imax"],c["sal_out"],c["best_c"],c["lo"],c["hi"],c["centre"])
        if c["has_o2"] and c["o2_out"] is not None and c["best_o2"] is not None:
            plot_oxygen(cast,c["theta"],c["P"],c["start"],c["imax"],c["o2_out"],c["best_o2"],c["lo"],c["hi"],c["centre"])
    print(f"\nplotted: {to_plot}")
    print(f"summary + plots -> {QC_PLOT_DIR}")


## 7. Findings and assumptions (for Leah)

- **Temperature advance +0.5 s** (19plus V2 row). Applied as the fixed baseline, not tuned.
- **Conductivity advance.** The manual/slides say the 19plus V2 needs no conductivity advance (the
  +0.5 s temperature advance handles the C-T alignment). Our QC on the strongly-stratified casts
  with a clean interior optimum leaned to a small **-0.15 s** conductivity advance, but the effect
  on salinity spiking is marginal (the dS/dt envelope barely changes; the advance mostly shifts the
  peak salinity a little). **Working assumption: keep conductivity = 0 s** unless the CTD_01 / CTD_03
  panels show a clear tightening at -0.15 s. Open question for Leah: adopt the small -0.15 s
  refinement, or keep 0?
- **Oxygen advance.** Within the recommended +3 to +7 s. The numeric optimum is unstable (pegs at the
  range edges on most casts), and the O2 profiles are noisy on the shelf, so the down/up plots do not
  cleanly pick a value. **Working assumption: keep oxygen raw = +5 s** (mid-range). Open question:
  does Leah read a preferred value from the O2-vs-potential-temperature panels?
- **Bad cast:** CTD_04 shows anomalous conductivity (dS/dt about 10x the others); exclude it and
  check its raw data.
- **The large down/up salinity difference at the interface is expected** (the slides flag it); the
  decision rests on the dS/dt spiking, not the down/up magnitude.
